In [9]:
import pandas as pd
import numpy as np

from catboost import CatBoostRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import warnings
warnings.filterwarnings('ignore')

In [11]:
df = pd.read_excel(r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\100\Preprocessing\100_Pre_done_Combined.xlsx")

df['Pstng Date'] = pd.to_datetime(df['Pstng Date'])

df.head()

,Material,SLoc,Quantity,Pstng Date,order,Equipment,Technician name,Year,Tavg,Tmax,Tmin,RH,Month,Season,Delta_T,Region,Location
0,100,5023,-1,2020-01-04,48550533,10930429,Anil Sharma,2020,12.94,21.27,7.26,58.52,1,Winter,14.01,North1,Haryana
1,100,5024,-1,2020-01-06,48556766,10844557,Jogendra Singh,2020,14.94,22.58,9.03,56.47,1,Winter,13.55,North1,Jaipur
2,100,5030,-1,2020-01-06,48550093,10517828,Himanshu Kushwaha,2020,13.70,21.92,8.25,55.23,1,Winter,13.67,North1,Delhi
3,100,5002,-1,2020-01-06,48550185,10519283,Prasad Chokhat,2020,22.83,30.49,16.34,68.83,1,Winter,14.15,West1,Navi Mumbai
4,100,5044,-1,2020-01-06,48554032,10836205,Joby Varghese,2020,28.71,32.68,25.08,68.80,1,Winter,7.60,South,Cochin


In [ ]:
monthly = (
    df.groupby(
        [
            pd.Grouper(key='Pstng Date', freq='MS'),
            'SLoc'
        ]
    )
    .agg(
        Consumption=('Quantity', lambda x: abs(x.sum())),
        Tavg=('Tavg', 'mean'),
        RH=('RH', 'mean'),
        Delta_T=('Delta_T', 'mean'),
        Region=('Region', 'first'),
        Location=('Location', 'first'),
        Season=('Season', 'first')
    )
    .reset_index()
)
monthly.head()

,Pstng Date,SLoc,Consumption,Tavg,RH,Delta_T,Region,Location,Season
0,2020-01-01,5001,19,21.661579,67.084211,13.224737,West1,Mumbai,Winter
1,2020-01-01,5002,4,22.910000,67.287500,11.530000,West1,Navi Mumbai,Winter
2,2020-01-01,5005,1,18.910000,54.610000,12.240000,West2,Ahmedabad,Winter
3,2020-01-01,5007,1,18.770000,74.430000,13.940000,West2,Raipur,Winter
4,2020-01-01,5010,2,21.490000,63.915000,14.455000,West2,Surat,Winter


In [13]:
monthly['Year'] = monthly['Pstng Date'].dt.year
monthly['Month'] = monthly['Pstng Date'].dt.month

monthly['sin_month'] = np.sin(
    2*np.pi*monthly['Month']/12
)

monthly['cos_month'] = np.cos(
    2*np.pi*monthly['Month']/12
)

In [14]:
monthly = monthly.sort_values(
    ['SLoc','Pstng Date']
)

for lag in [1,2,3,6,12]:

    monthly[f'lag_{lag}'] = (
        monthly
        .groupby('SLoc')['Consumption']
        .shift(lag)
    )

In [15]:
monthly['roll_3'] = (
    monthly
    .groupby('SLoc')['Consumption']
    .transform(
        lambda x:
        x.shift(1).rolling(3).mean()
    )
)

monthly['roll_6'] = (
    monthly
    .groupby('SLoc')['Consumption']
    .transform(
        lambda x:
        x.shift(1).rolling(6).mean()
    )
)

In [16]:
monthly = monthly.dropna().reset_index(drop=True)

monthly.shape

(849, 20)

In [17]:
split_date = '2025-01-01'

train = monthly[
    monthly['Pstng Date'] < split_date
]

val = monthly[
    monthly['Pstng Date'] >= split_date
]

In [18]:
features = [
    'SLoc',
    'Region',
    'Location',
    'Season',

    'Tavg',
    'RH',
    'Delta_T',

    'Month',
    'Year',

    'sin_month',
    'cos_month',

    'lag_1',
    'lag_2',
    'lag_3',
    'lag_6',
    'lag_12',

    'roll_3',
    'roll_6'
]

target = 'Consumption'

In [19]:
cat_features = [
    'SLoc',
    'Region',
    'Location',
    'Season'
]

model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.03,
    depth=8,
    loss_function='RMSE',
    verbose=100
)

model.fit(
    train[features],
    train[target],
    cat_features=cat_features
)

0:	learn: 23.7455578	total: 188ms	remaining: 3m 8s
100:	learn: 7.2295209	total: 8s	remaining: 1m 11s
200:	learn: 4.7124528	total: 14.7s	remaining: 58.6s
300:	learn: 3.7304082	total: 21.2s	remaining: 49.2s
400:	learn: 3.0647180	total: 27.9s	remaining: 41.7s
500:	learn: 2.5261586	total: 34.6s	remaining: 34.5s
600:	learn: 2.1707542	total: 41.2s	remaining: 27.4s
700:	learn: 1.8926255	total: 47.6s	remaining: 20.3s
800:	learn: 1.6371688	total: 54.1s	remaining: 13.4s
900:	learn: 1.4250161	total: 1m	remaining: 6.63s
999:	learn: 1.2555739	total: 1m 6s	remaining: 0us


CatBoostRegressor(depth=8, iterations=1000, learning_rate=0.03, loss_function='RMSE', verbose=100)

In [ ]:
val_pred = model.predict(
    val[features]
)

mae = mean_absolute_error(
    val[target],
    val_pred
)

rmse = np.sqrt(
    mean_squared_error(
        val[target],
        val_pred
    )
)

mape = np.mean(
    np.abs(
        (
            val[target]
            - val_pred
        )
        /
        val[target]
    )
) * 100

print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)

MAE : 5.440987785464996
RMSE: 11.326271240903717
MAPE: inf


In [21]:
forecast_pivot = forecast_df.pivot(
    index='SLoc',
    columns='Month_Name',
    values='Forecast'
)

forecast_pivot.to_excel(
    'SLoc_Forecast.xlsx'
)

NameError: name 'forecast_df' is not defined

In [22]:
import requests
import pandas as pd

LAT = 28.6139
LON = 77.2090

url = (
    "https://api.open-meteo.com/v1/forecast"
    f"?latitude={LAT}"
    f"&longitude={LON}"
    "&daily=temperature_2m_max,"
    "temperature_2m_min,"
    "temperature_2m_mean,"
    "relative_humidity_2m_mean"
    "&timezone=auto"
    "&forecast_days=16"
)

response = requests.get(url)

data = response.json()

weather_df = pd.DataFrame({
    'Date': data['daily']['time'],
    'Tmax': data['daily']['temperature_2m_max'],
    'Tmin': data['daily']['temperature_2m_min'],
    'Tavg': data['daily']['temperature_2m_mean'],
    'RH': data['daily']['relative_humidity_2m_mean']
})

weather_df['Delta_T'] = (
    weather_df['Tmax']
    - weather_df['Tmin']
)

print(weather_df.head())


         Date  Tmax  Tmin  Tavg  RH  Delta_T
0  2026-06-02  36.2  24.4  30.8  52     11.8
1  2026-06-03  39.7  28.0  33.7  43     11.7
2  2026-06-04  40.7  29.5  35.0  36     11.2
3  2026-06-05  38.2  29.5  33.8  41      8.7
4  2026-06-06  36.0  27.9  32.2  50      8.1
